# Regional Electricity Cost Analysis: High-Cost Northeast & Appalachian IOUs

Analysis of revenue requirements, O&M costs, rate base, and financial metrics for
Investor-Owned Utilities (IOUs) in high electricity cost states:
**Maine, New York, Massachusetts, West Virginia, and Maryland**.

Data source: PUDL FERC Form 1 (2015-2024), accessed via the nightly S3 parquet builds.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from matplotlib.patches import Patch

S3_BASE = "https://s3.us-west-2.amazonaws.com/pudl.catalyst.coop/nightly"

YEARS = range(2015, 2025)
BASE_YEAR = 2015
REAL_DOLLAR_YEAR = 2024

# BLS CPI-U Annual Averages (All Items, U.S. City Average, 1982-84=100, NSA)
# Source: https://www.bls.gov/cpi/tables/supplemental-files/
CPI_U = {
    2015: 237.017,
    2016: 240.007,
    2017: 245.120,
    2018: 251.107,
    2019: 255.657,
    2020: 258.811,
    2021: 270.970,
    2022: 292.655,
    2023: 304.702,
    2024: 313.689,
}

# Deflator: multiply nominal $ by this to get real REAL_DOLLAR_YEAR $
DEFLATOR = {yr: CPI_U[REAL_DOLLAR_YEAR] / cpi for yr, cpi in CPI_U.items()}

# Minimum annual retail sales (MWh) to include a utility in per-MWh calculations.
# Filters out T&D-only utilities in deregulated states (e.g., Central Maine Power)
# that report near-zero commodity sales but significant distribution revenue.
MIN_RETAIL_MWH = 100_000


def deflate(df, dollar_cols, year_col="report_year"):
    """Convert nominal dollar columns to real (REAL_DOLLAR_YEAR) dollars using CPI-U."""
    df = df.copy()
    for col in dollar_cols:
        if col in df.columns:
            df[col] = df[col] * df[year_col].map(DEFLATOR)
    return df

## 1. Identify IOUs in Target States

We select major IOUs that serve customers in ME, NY, MA, WV, and MD from the FERC Form 1 respondents.

In [ ]:
ferc1_utils = pd.read_parquet(f"{S3_BASE}/core_pudl__assn_ferc1_pudl_utilities.parquet")

# Major IOUs in each target state (utility_id_ferc1 -> name, state)
TARGET_UTILITIES = {
    # Maine
    212: {"name": "Central Maine Power", "state": "ME", "color": "#1f77b4"},
    158: {"name": "Emera Maine (Versant)", "state": "ME", "color": "#aec7e8"},
    # New York
    159: {"name": "Consolidated Edison", "state": "NY", "color": "#ff7f0e"},
    246: {"name": "Central Hudson G&E", "state": "NY", "color": "#ffbb78"},
    215: {"name": "NY State Electric & Gas", "state": "NY", "color": "#d62728"},
    275: {"name": "Niagara Mohawk (National Grid)", "state": "NY", "color": "#e377c2"},
    214: {"name": "Rochester Gas & Electric", "state": "NY", "color": "#ff9896"},
    190: {"name": "Orange & Rockland Utilities", "state": "NY", "color": "#c49c94"},
    # Massachusetts
    277: {"name": "Massachusetts Electric (National Grid)", "state": "MA", "color": "#2ca02c"},
    271: {"name": "NSTAR Electric (Eversource)", "state": "MA", "color": "#98df8a"},
    245: {"name": "Western Mass Electric (Eversource)", "state": "MA", "color": "#7f7f7f"},
    # West Virginia
    238: {"name": "Monongahela Power", "state": "WV", "color": "#9467bd"},
    200: {"name": "Appalachian Power", "state": "WV", "color": "#c5b0d5"},
    240: {"name": "Potomac Edison", "state": "WV", "color": "#8c564b"},
    # Maryland
    248: {"name": "Baltimore Gas & Electric", "state": "MD", "color": "#bcbd22"},
    290: {"name": "Potomac Electric Power (Pepco)", "state": "MD", "color": "#dbdb8d"},
    291: {"name": "Delmarva Power & Light", "state": "MD", "color": "#17becf"},
}

UTIL_IDS = list(TARGET_UTILITIES.keys())
UTIL_NAMES = {uid: info["name"] for uid, info in TARGET_UTILITIES.items()}
UTIL_COLORS = {uid: info["color"] for uid, info in TARGET_UTILITIES.items()}
UTIL_STATES = {uid: info["state"] for uid, info in TARGET_UTILITIES.items()}

# Group by state for state-level analysis
STATE_UTILS = {}
for uid, info in TARGET_UTILITIES.items():
    STATE_UTILS.setdefault(info["state"], []).append(uid)

STATE_COLORS = {"ME": "#1f77b4", "NY": "#ff7f0e", "MA": "#2ca02c", "WV": "#9467bd", "MD": "#bcbd22"}
STATE_NAMES = {"ME": "Maine", "NY": "New York", "MA": "Massachusetts", "WV": "West Virginia", "MD": "Maryland"}

print(f"Analyzing {len(TARGET_UTILITIES)} IOUs across {len(STATE_UTILS)} states")
for st, uids in STATE_UTILS.items():
    print(f"  {STATE_NAMES[st]}: {', '.join(UTIL_NAMES[u] for u in uids)}")

In [ ]:
plt.rcParams['font.size'] = 14
plt.rcParams['axes.labelsize'] = 16
plt.rcParams['axes.labelweight'] = 'bold'
plt.rcParams['xtick.labelsize'] = 13
plt.rcParams['ytick.labelsize'] = 13
plt.rcParams['legend.fontsize'] = 11
plt.rcParams['figure.titlesize'] = 18
plt.rcParams['axes.xmargin'] = 0.02

## 2. Load FERC Form 1 Data from PUDL

In [ ]:
# Operating Expenses (Schedule 320)
opex_all = pd.read_parquet(f"{S3_BASE}/core_ferc1__yearly_operating_expenses_sched320.parquet")
opex = opex_all[
    (opex_all["utility_id_ferc1"].isin(UTIL_IDS)) &
    (opex_all["report_year"].isin(YEARS))
].copy()
opex = deflate(opex, ["dollar_value"])
print(f"Operating Expenses: {len(opex)} rows")

# Plant in Service (Schedule 204) - for rate base proxy
plant_all = pd.read_parquet(f"{S3_BASE}/core_ferc1__yearly_plant_in_service_sched204.parquet")
plant = plant_all[
    (plant_all["utility_id_ferc1"].isin(UTIL_IDS)) &
    (plant_all["report_year"].isin(YEARS))
].copy()
plant = deflate(plant, ["ending_balance", "additions", "retirements", "adjustments", "transfers"])
print(f"Plant in Service: {len(plant)} rows")

# Rate Base (PUDL output table - combines Schedules 110, 118, 200, 204, 219, 320)
rate_base_all = pd.read_parquet(f"{S3_BASE}/out_ferc1__yearly_rate_base.parquet")
rate_base_raw = rate_base_all[
    (rate_base_all["utility_id_ferc1"].isin(UTIL_IDS)) &
    (rate_base_all["report_year"].isin(YEARS))
].copy()
rate_base_raw = deflate(rate_base_raw, ["ending_balance"])
print(f"Rate Base (output table): {len(rate_base_raw)} rows")

# Operating Revenues (Schedule 300)
rev_all = pd.read_parquet(f"{S3_BASE}/core_ferc1__yearly_operating_revenues_sched300.parquet")
rev = rev_all[
    (rev_all["utility_id_ferc1"].isin(UTIL_IDS)) &
    (rev_all["report_year"].isin(YEARS))
].copy()
rev = deflate(rev, ["dollar_value"])
print(f"Operating Revenues: {len(rev)} rows")

# Income Statements (Schedule 114)
inc_all = pd.read_parquet(f"{S3_BASE}/core_ferc1__yearly_income_statements_sched114.parquet")
inc = inc_all[
    (inc_all["utility_id_ferc1"].isin(UTIL_IDS)) &
    (inc_all["report_year"].isin(YEARS))
].copy()
inc = deflate(inc, ["dollar_value"])
print(f"Income Statements: {len(inc)} rows")

# Depreciation Summary (Schedule 336)
dep_all = pd.read_parquet(f"{S3_BASE}/core_ferc1__yearly_depreciation_summary_sched336.parquet")
dep = dep_all[
    (dep_all["utility_id_ferc1"].isin(UTIL_IDS)) &
    (dep_all["report_year"].isin(YEARS))
].copy()
dep = deflate(dep, ["dollar_value"])
print(f"Depreciation Summary: {len(dep)} rows")

# Sales by Rate Schedule (Schedule 304) - for MWh sales data
sales_all = pd.read_parquet(f"{S3_BASE}/core_ferc1__yearly_sales_by_rate_schedules_sched304.parquet")
sales = sales_all[
    (sales_all["utility_id_ferc1"].isin(UTIL_IDS)) &
    (sales_all["report_year"].isin(YEARS))
].copy()
print(f"Sales by Rate Schedule: {len(sales)} rows")

print(f"\nAll dollar values deflated to {REAL_DOLLAR_YEAR} dollars using CPI-U.")

## 3. O&M Expenses by Functional Category

Extract generation, transmission, and distribution O&M expenses from Schedule 320.

In [ ]:
# Map expense_type to functional categories
# Use the top-level subtotals reported in Form 1
OPEX_CATEGORIES = {
    "generation_expenses": "Generation",
    "purchased_power": "Purchased Power",
    "transmission_expenses": "Transmission",
    "distribution_expenses": "Distribution",
    "customer_account_expenses": "Customer Accounts",
    "customer_service_and_information_expenses": "Customer Service",
    "sales_expenses": "Sales",
    "administrative_and_general_expenses": "Admin & General",
}

opex_cat = opex[opex["expense_type"].isin(OPEX_CATEGORIES.keys())].copy()
opex_cat["category"] = opex_cat["expense_type"].map(OPEX_CATEGORIES)
opex_cat["utility_name"] = opex_cat["utility_id_ferc1"].map(UTIL_NAMES)
opex_cat["state"] = opex_cat["utility_id_ferc1"].map(UTIL_STATES)

# Pivot to get year x utility x category
opex_pivot = opex_cat.pivot_table(
    index=["utility_id_ferc1", "utility_name", "state", "report_year"],
    columns="category",
    values="dollar_value",
    aggfunc="sum"
).reset_index()

# Create combined "Power Supply" = Generation O&M + Purchased Power
opex_pivot["Power Supply"] = (
    opex_pivot["Generation"].fillna(0) + opex_pivot["Purchased Power"].fillna(0)
)

# Also get total O&M
total_opex = opex[
    opex["expense_type"] == "operations_and_maintenance_expenses_electric"
].copy()
total_opex["utility_name"] = total_opex["utility_id_ferc1"].map(UTIL_NAMES)
total_opex["state"] = total_opex["utility_id_ferc1"].map(UTIL_STATES)

print("O&M data extracted (including Purchased Power).")
print(f"Utilities with data: {opex_pivot['utility_name'].nunique()}")
print(f"Years: {sorted(opex_pivot['report_year'].unique())}")
opex_pivot.head()

In [ ]:
# Break generation O&M into sub-categories
GEN_SUBCATS = {
    "steam_power_generation_operations_expense": "Steam Operations",
    "steam_power_generation_maintenance_expense": "Steam Maintenance",
    "nuclear_power_generation_operations_expense": "Nuclear Operations",
    "nuclear_power_generation_maintenance_expense": "Nuclear Maintenance",
    "hydraulic_power_generation_operations_expense": "Hydro Operations",
    "hydraulic_power_generation_maintenance_expense": "Hydro Maintenance",
    "other_power_generation_operations_expense": "Other Gen Operations",
    "other_power_generation_maintenance_expense": "Other Gen Maintenance",
    "purchased_power": "Purchased Power",
    "other_power_supply_expense": "Other Power Supply",
}

gen_detail = opex[opex["expense_type"].isin(GEN_SUBCATS.keys())].copy()
gen_detail["subcategory"] = gen_detail["expense_type"].map(GEN_SUBCATS)
gen_detail["utility_name"] = gen_detail["utility_id_ferc1"].map(UTIL_NAMES)
gen_detail["state"] = gen_detail["utility_id_ferc1"].map(UTIL_STATES)

### 3a. O&M Expenses Over Time (Real 2024 $)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(24, 16), sharey=False)

plot_cats = [
    ("Power Supply", axes[0, 0]),
    ("Purchased Power", axes[0, 1]),
    ("Transmission", axes[1, 0]),
    ("Distribution", axes[1, 1]),
]

for cat, ax in plot_cats:
    for uid in UTIL_IDS:
        df_u = opex_pivot[opex_pivot["utility_id_ferc1"] == uid].sort_values("report_year")
        if cat in df_u.columns and df_u[cat].notna().any():
            ax.plot(df_u["report_year"], df_u[cat] / 1e9,
                    color=UTIL_COLORS[uid], lw=2.5, label=UTIL_NAMES[uid])
    subtitle = "(Generation O&M + Purchased Power)" if cat == "Power Supply" else ""
    ax.set_title(f"{cat} Expenses{chr(10)}{subtitle}".strip(), fontweight="bold")
    ax.set_xlabel("Year")
    ax.set_ylabel("Billion 2024 $")
    ax.grid(lw=0.3)
    ax.xaxis.set_major_locator(mticker.MaxNLocator(integer=True))

axes[0, 0].legend(bbox_to_anchor=(0, -0.25), loc="upper left", ncol=3, fontsize=9)
plt.suptitle("O&M Expenses by Functional Category (Real 2024 $)\nFERC Form 1, 2015-2024", fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

### 3b. O&M Expenses Normalized to Base Year (2015 = 1.0)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(24, 16), sharey=True)

plot_cats = [
    ("Power Supply", axes[0, 0]),
    ("Purchased Power", axes[0, 1]),
    ("Transmission", axes[1, 0]),
    ("Distribution", axes[1, 1]),
]

for cat, ax in plot_cats:
    for uid in UTIL_IDS:
        df_u = opex_pivot[opex_pivot["utility_id_ferc1"] == uid].sort_values("report_year")
        if cat not in df_u.columns or df_u[cat].isna().all():
            continue
        base_val = df_u[df_u["report_year"] == BASE_YEAR][cat]
        if base_val.empty or base_val.item() == 0:
            continue
        normalized = df_u[cat] / base_val.item()
        ax.plot(df_u["report_year"], normalized,
                color=UTIL_COLORS[uid], lw=2.5, label=UTIL_NAMES[uid])
    ax.axhline(y=1, linestyle="dashed", color="grey", lw=1)
    subtitle = "(Gen O&M + Purchased Power)" if cat == "Power Supply" else ""
    ax.set_title(f"{cat} (Normalized){chr(10)}{subtitle}".strip(), fontweight="bold")
    ax.set_xlabel("Year")
    ax.set_ylabel(f"Relative to {BASE_YEAR}")
    ax.grid(lw=0.3)
    ax.xaxis.set_major_locator(mticker.MaxNLocator(integer=True))

axes[0, 0].legend(bbox_to_anchor=(0, -0.25), loc="upper left", ncol=3, fontsize=9)
plt.suptitle(f"O&M Expenses Normalized ({BASE_YEAR} = 1.0)\nFERC Form 1", fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

### 3c. Total O&M by State (Aggregated)

In [ ]:
# Aggregate O&M categories by state
state_opex = opex_pivot.groupby(["state", "report_year"])[
    ["Generation", "Purchased Power", "Power Supply", "Transmission", "Distribution", "Admin & General"]
].sum().reset_index()

fig, axes = plt.subplots(2, 2, figsize=(24, 16), sharey=False)

plot_cats = [
    ("Power Supply", axes[0, 0]),
    ("Purchased Power", axes[0, 1]),
    ("Transmission", axes[1, 0]),
    ("Distribution", axes[1, 1]),
]

for cat, ax in plot_cats:
    for st in STATE_COLORS:
        df_s = state_opex[state_opex["state"] == st].sort_values("report_year")
        if cat in df_s.columns and df_s[cat].notna().any():
            ax.plot(df_s["report_year"], df_s[cat] / 1e9,
                    color=STATE_COLORS[st], lw=3, label=STATE_NAMES[st])
    subtitle = "(Gen O&M + Purchased Power)" if cat == "Power Supply" else ""
    ax.set_title(f"{cat} by State{chr(10)}{subtitle}".strip(), fontweight="bold")
    ax.set_xlabel("Year")
    ax.set_ylabel("Billion 2024 $")
    ax.grid(lw=0.3)
    ax.legend()
    ax.xaxis.set_major_locator(mticker.MaxNLocator(integer=True))

plt.suptitle("State-Aggregated O&M Expenses (Real 2024 $)\nFERC Form 1, 2015-2024", fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

### 3d. O&M Cost Breakdown (Stacked Bar) per Utility

In [ ]:
# Stacked bar for most recent year
latest_year = opex_pivot["report_year"].max()
latest = opex_pivot[opex_pivot["report_year"] == latest_year].copy()
latest = latest.set_index("utility_name")

stack_cols = ["Generation", "Purchased Power", "Transmission", "Distribution",
              "Admin & General", "Customer Accounts", "Customer Service", "Sales"]
stack_cols = [c for c in stack_cols if c in latest.columns]

fig, ax = plt.subplots(figsize=(16, 8))
latest[stack_cols].div(1e9).plot(kind="barh", stacked=True, ax=ax,
                                 colormap="tab10", edgecolor="white")
ax.set_xlabel("Billion 2024 $")
ax.set_title(f"O&M Cost Breakdown by Utility ({latest_year})\nFERC Form 1 Schedule 320", fontweight="bold")
ax.legend(loc="lower right")
ax.grid(axis="x", lw=0.3)
plt.tight_layout()
plt.show()

## 4. Rate Base / Plant in Service

Using Schedule 204 (Electric Plant in Service) to analyze the rate base
by functional category: production, transmission, distribution, and general plant.

In [ ]:
# Map ferc_account_label to functional categories for plant in service
PLANT_CATEGORIES = {
    "production_plant": "Production",
    "steam_production_plant": "Steam Production",
    "nuclear_production_plant": "Nuclear Production",
    "hydraulic_production_plant": "Hydro Production",
    "other_production_plant": "Other Production",
    "transmission_plant": "Transmission",
    "distribution_plant": "Distribution",
    "general_plant": "General",
    "intangible_plant": "Intangible",
    "electric_plant_in_service": "Total Electric Plant",
}

# Filter to in_service plant status and the top-level categories
plant_cat = plant[
    (plant["ferc_account_label"].isin(PLANT_CATEGORIES.keys())) &
    (plant["plant_status"] == "in_service")
].copy()
plant_cat["category"] = plant_cat["ferc_account_label"].map(PLANT_CATEGORIES)
plant_cat["utility_name"] = plant_cat["utility_id_ferc1"].map(UTIL_NAMES)
plant_cat["state"] = plant_cat["utility_id_ferc1"].map(UTIL_STATES)

# Pivot: ending_balance is the net plant in service
plant_pivot = plant_cat.pivot_table(
    index=["utility_id_ferc1", "utility_name", "state", "report_year"],
    columns="category",
    values="ending_balance",
    aggfunc="sum"
).reset_index()

print(f"Plant in service data: {len(plant_pivot)} rows")
plant_pivot.head()

### 4a. Plant in Service (Rate Base Proxy) Over Time

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(24, 8), sharey=False)

for ax, cat in zip(axes, ["Production", "Transmission", "Distribution"]):
    for uid in UTIL_IDS:
        df_u = plant_pivot[plant_pivot["utility_id_ferc1"] == uid].sort_values("report_year")
        if cat in df_u.columns and df_u[cat].notna().any():
            ax.plot(df_u["report_year"], df_u[cat] / 1e9,
                    color=UTIL_COLORS[uid], lw=2.5, label=UTIL_NAMES[uid])
    ax.set_title(f"{cat} Plant in Service", fontweight="bold")
    ax.set_xlabel("Year")
    ax.set_ylabel("Billion 2024 $")
    ax.grid(lw=0.3)
    ax.xaxis.set_major_locator(mticker.MaxNLocator(integer=True))

axes[0].legend(bbox_to_anchor=(0, -0.25), loc="upper left", ncol=3, fontsize=9)
plt.suptitle("Electric Plant in Service (Real 2024 $)\nFERC Form 1 Schedule 204, 2015-2024", fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

### 4b. Plant in Service Normalized (2015 = 1.0)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(24, 8), sharey=True)

for ax, cat in zip(axes, ["Production", "Transmission", "Distribution"]):
    for uid in UTIL_IDS:
        df_u = plant_pivot[plant_pivot["utility_id_ferc1"] == uid].sort_values("report_year")
        if cat not in df_u.columns or df_u[cat].isna().all():
            continue
        base_val = df_u[df_u["report_year"] == BASE_YEAR][cat]
        if base_val.empty or base_val.item() == 0:
            continue
        normalized = df_u[cat] / base_val.item()
        ax.plot(df_u["report_year"], normalized,
                color=UTIL_COLORS[uid], lw=2.5, label=UTIL_NAMES[uid])
    ax.axhline(y=1, linestyle="dashed", color="grey", lw=1)
    ax.set_title(f"{cat} Plant (Normalized)", fontweight="bold")
    ax.set_xlabel("Year")
    ax.set_ylabel(f"Relative to {BASE_YEAR}")
    ax.grid(lw=0.3)
    ax.xaxis.set_major_locator(mticker.MaxNLocator(integer=True))

axes[0].legend(bbox_to_anchor=(0, -0.25), loc="upper left", ncol=3, fontsize=9)
plt.suptitle(f"Plant in Service Normalized ({BASE_YEAR} = 1.0)\nFERC Form 1 Schedule 204", fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

### 4c. Capital Additions by Category

In [ ]:
# Additions pivot
add_pivot = plant_cat.pivot_table(
    index=["utility_id_ferc1", "utility_name", "state", "report_year"],
    columns="category",
    values="additions",
    aggfunc="sum"
).reset_index()

fig, axes = plt.subplots(1, 3, figsize=(24, 8), sharey=False)

for ax, cat in zip(axes, ["Production", "Transmission", "Distribution"]):
    for uid in UTIL_IDS:
        df_u = add_pivot[add_pivot["utility_id_ferc1"] == uid].sort_values("report_year")
        if cat in df_u.columns and df_u[cat].notna().any():
            ax.plot(df_u["report_year"], df_u[cat] / 1e9,
                    color=UTIL_COLORS[uid], lw=2.5, label=UTIL_NAMES[uid])
    ax.set_title(f"{cat} Capital Additions", fontweight="bold")
    ax.set_xlabel("Year")
    ax.set_ylabel("Billion 2024 $")
    ax.grid(lw=0.3)
    ax.xaxis.set_major_locator(mticker.MaxNLocator(integer=True))

axes[0].legend(bbox_to_anchor=(0, -0.25), loc="upper left", ncol=3, fontsize=9)
plt.suptitle("Annual Capital Additions (Real 2024 $)\nFERC Form 1 Schedule 204, 2015-2024", fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

### 4d. Actual Rate Base (PUDL Output Table)

The `out_ferc1__yearly_rate_base` table provides a proper rate base calculation that accounts for
accumulated depreciation, CWIP, ADIT, and working capital — unlike the gross plant-in-service proxy above.

This table was developed in collaboration with RMI and combines data from FERC Form 1 Schedules 110, 118, 200, 204, 219, and 320.

In [ ]:
# Map rate_base_category to functional groups for comparison with plant-in-service
RATE_BASE_FUNC_MAP = {
    "steam": "Production",
    "nuclear": "Production",
    "hydro": "Production",
    "other_production": "Production",
    "transmission": "Transmission",
    "distribution": "Distribution",
    "general_plant": "General",
    "intangible_plant": "Intangible",
    "net_ADIT": "Net ADIT",
    "net_working_capital": "Net Working Capital",
    "net_utility_plant": "Net Utility Plant Adjustments",
    "net_nuclear_fuel": "Production",
    "other_plant": "Other",
    "experimental_plant": "Other",
    "regional_transmission_and_market_operation": "Transmission",
    "net_regulatory_assets": "Regulatory Assets",
    "other_deferred_debits_and_credits": "Other",
    "AROs": "Other",
    "asset_retirement_costs": "Other",
    "utility_plant": "Net Utility Plant Adjustments",
}

# Filter to electric utility type and aggregate by functional group
rb = rate_base_raw[rate_base_raw["utility_type"] == "electric"].copy()
rb["func_group"] = rb["rate_base_category"].map(RATE_BASE_FUNC_MAP)
rb["utility_name"] = rb["utility_id_ferc1"].map(UTIL_NAMES)
rb["state"] = rb["utility_id_ferc1"].map(UTIL_STATES)

# Aggregate: total rate base per utility per year per functional group
rb_func = (
    rb.groupby(["utility_id_ferc1", "utility_name", "state", "report_year", "func_group"])[
        "ending_balance"
    ]
    .sum()
    .reset_index()
)

rb_pivot = rb_func.pivot_table(
    index=["utility_id_ferc1", "utility_name", "state", "report_year"],
    columns="func_group",
    values="ending_balance",
    aggfunc="sum",
).reset_index()

# Compute total rate base (sum of all components, ADIT subtracts automatically since it's negative)
rate_base_components = [
    c for c in rb_pivot.columns
    if c not in ["utility_id_ferc1", "utility_name", "state", "report_year"]
]
rb_pivot["Total Rate Base"] = rb_pivot[rate_base_components].sum(axis=1)

print(f"Rate base data: {len(rb_pivot)} utility-year rows")
print(f"Functional groups: {sorted(rate_base_components)}")
rb_pivot.head()

In [ ]:
# Rate base by function (absolute)
fig, axes = plt.subplots(1, 3, figsize=(24, 8), sharey=False)
for ax, cat in zip(axes, ["Production", "Transmission", "Distribution"]):
    for uid in UTIL_IDS:
        df_u = rb_pivot[rb_pivot["utility_id_ferc1"] == uid].sort_values("report_year")
        if cat in df_u.columns and df_u[cat].notna().any():
            ax.plot(
                df_u["report_year"], df_u[cat] / 1e9,
                color=UTIL_COLORS[uid], lw=2.5, label=UTIL_NAMES[uid],
            )
    ax.set_title(f"{cat} Rate Base", fontweight="bold")
    ax.set_xlabel("Year")
    ax.set_ylabel("Billion 2024 $")
    ax.grid(lw=0.3)
    ax.xaxis.set_major_locator(mticker.MaxNLocator(integer=True))

axes[0].legend(bbox_to_anchor=(0, -0.25), loc="upper left", ncol=3, fontsize=9)
plt.suptitle(
    "Rate Base by Function (Real 2024 $)\nFERC Form 1 (PUDL Rate Base Output), 2015-2024",
    fontweight="bold", y=1.02,
)
plt.tight_layout()
plt.show()

In [ ]:
# Rate base normalized (base year = 1.0)
fig, axes = plt.subplots(1, 3, figsize=(24, 8), sharey=True)
for ax, cat in zip(axes, ["Production", "Transmission", "Distribution"]):
    for uid in UTIL_IDS:
        df_u = rb_pivot[rb_pivot["utility_id_ferc1"] == uid].sort_values("report_year")
        if cat not in df_u.columns or df_u[cat].isna().all():
            continue
        base_val = df_u[df_u["report_year"] == BASE_YEAR][cat]
        if base_val.empty or base_val.item() == 0:
            continue
        normalized = df_u[cat] / base_val.item()
        ax.plot(
            df_u["report_year"], normalized,
            color=UTIL_COLORS[uid], lw=2.5, label=UTIL_NAMES[uid],
        )
    ax.axhline(y=1, linestyle="dashed", color="grey", lw=1)
    ax.set_title(f"{cat} Rate Base (Normalized)", fontweight="bold")
    ax.set_xlabel("Year")
    ax.set_ylabel(f"Relative to {BASE_YEAR}")
    ax.grid(lw=0.3)
    ax.xaxis.set_major_locator(mticker.MaxNLocator(integer=True))

axes[0].legend(bbox_to_anchor=(0, -0.25), loc="upper left", ncol=3, fontsize=9)
plt.suptitle(
    f"Rate Base Normalized ({BASE_YEAR} = 1.0)\nFERC Form 1 (PUDL Rate Base Output)",
    fontweight="bold", y=1.02,
)
plt.tight_layout()
plt.show()

### 4e. Rate Base Proxy Comparison: Plant in Service vs Actual Rate Base

Gross plant in service (Schedule 204) overstates the rate base because it doesn't subtract accumulated
depreciation, ADIT, or account for working capital. How much does the proxy overstate?

In [ ]:
from matplotlib.lines import Line2D

fig, axes = plt.subplots(1, 2, figsize=(20, 8))

# Left: Total plant in service vs total rate base (absolute)
ax = axes[0]
for uid in UTIL_IDS:
    df_plant = plant_pivot[plant_pivot["utility_id_ferc1"] == uid].sort_values("report_year")
    df_rb = rb_pivot[rb_pivot["utility_id_ferc1"] == uid].sort_values("report_year")
    if "Total Electric Plant" in df_plant.columns and len(df_plant) > 0:
        ax.plot(
            df_plant["report_year"], df_plant["Total Electric Plant"] / 1e9,
            color=UTIL_COLORS[uid], lw=2, linestyle="--", alpha=0.6,
        )
    if "Total Rate Base" in df_rb.columns and len(df_rb) > 0:
        ax.plot(
            df_rb["report_year"], df_rb["Total Rate Base"] / 1e9,
            color=UTIL_COLORS[uid], lw=2.5,
        )

proxy_handles = [
    Line2D([0], [0], color="black", lw=2.5, label="Actual Rate Base"),
    Line2D([0], [0], color="black", lw=2, linestyle="--", alpha=0.6, label="Plant in Service (proxy)"),
]
ax.legend(handles=proxy_handles, fontsize=11)
ax.set_title("Total Rate Base vs Plant in Service Proxy", fontweight="bold")
ax.set_xlabel("Year")
ax.set_ylabel("Billion 2024 $")
ax.grid(lw=0.3)
ax.xaxis.set_major_locator(mticker.MaxNLocator(integer=True))

# Right: Ratio of rate base to plant in service
ax = axes[1]
for uid in UTIL_IDS:
    df_plant = plant_pivot[plant_pivot["utility_id_ferc1"] == uid].sort_values("report_year")
    df_rb = rb_pivot[rb_pivot["utility_id_ferc1"] == uid].sort_values("report_year")
    if "Total Electric Plant" not in df_plant.columns or "Total Rate Base" not in df_rb.columns:
        continue
    merged = df_plant[["utility_id_ferc1", "report_year", "Total Electric Plant"]].merge(
        df_rb[["utility_id_ferc1", "report_year", "Total Rate Base"]],
        on=["utility_id_ferc1", "report_year"],
    )
    merged = merged[merged["Total Electric Plant"] > 0]
    if len(merged) > 0:
        ratio = merged["Total Rate Base"] / merged["Total Electric Plant"]
        ax.plot(
            merged["report_year"], ratio,
            color=UTIL_COLORS[uid], lw=2.5, label=UTIL_NAMES[uid],
        )

ax.axhline(y=1, linestyle="dashed", color="grey", lw=1)
ax.set_title("Rate Base / Plant in Service Ratio", fontweight="bold")
ax.set_xlabel("Year")
ax.set_ylabel("Ratio")
ax.grid(lw=0.3)
ax.legend(fontsize=8, ncol=2)
ax.xaxis.set_major_locator(mticker.MaxNLocator(integer=True))

plt.suptitle(
    "Rate Base Proxy Comparison\nPlant in Service (Sched 204) vs PUDL Rate Base Output",
    fontweight="bold", y=1.02,
)
plt.tight_layout()
plt.show()

## 5. Revenue & Electricity Sales

Using Schedules 300 and 304 for operating revenues and retail sales.

In [ ]:
# Total operating revenues
total_rev = rev[rev["revenue_type"] == "electric_operating_revenues"].copy()
total_rev["utility_name"] = total_rev["utility_id_ferc1"].map(UTIL_NAMES)
total_rev["state"] = total_rev["utility_id_ferc1"].map(UTIL_STATES)

# Revenue by customer class
REV_CLASSES = {
    "residential_sales": "Residential",
    "small_or_commercial": "Commercial",
    "large_or_industrial": "Industrial",
    "sales_to_ultimate_consumers": "Total Retail",
    "sales_for_resale": "Wholesale",
}

rev_class = rev[rev["revenue_type"].isin(REV_CLASSES.keys())].copy()
rev_class["customer_class"] = rev_class["revenue_type"].map(REV_CLASSES)
rev_class["utility_name"] = rev_class["utility_id_ferc1"].map(UTIL_NAMES)
rev_class["state"] = rev_class["utility_id_ferc1"].map(UTIL_STATES)

print(f"Total revenue records: {len(total_rev)}")
print(f"Revenue by class records: {len(rev_class)}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(20, 8))

# Total revenue
ax = axes[0]
for uid in UTIL_IDS:
    df_u = total_rev[total_rev["utility_id_ferc1"] == uid].sort_values("report_year")
    if len(df_u) > 0:
        ax.plot(df_u["report_year"], df_u["dollar_value"] / 1e9,
                color=UTIL_COLORS[uid], lw=2.5, label=UTIL_NAMES[uid])
ax.set_title("Total Electric Operating Revenues", fontweight="bold")
ax.set_xlabel("Year")
ax.set_ylabel("Billion 2024 $")
ax.grid(lw=0.3)
ax.legend(fontsize=8, ncol=2)
ax.xaxis.set_major_locator(mticker.MaxNLocator(integer=True))

# Revenue by class for total retail sales (MWh) where available
ax = axes[1]
retail = rev_class[rev_class["customer_class"] == "Total Retail"]
for uid in UTIL_IDS:
    df_u = retail[retail["utility_id_ferc1"] == uid].sort_values("report_year")
    if len(df_u) > 0 and df_u["sales_mwh"].notna().any():
        ax.plot(df_u["report_year"], df_u["sales_mwh"] / 1e6,
                color=UTIL_COLORS[uid], lw=2.5, label=UTIL_NAMES[uid])
ax.set_title("Retail Electricity Sales", fontweight="bold")
ax.set_xlabel("Year")
ax.set_ylabel("Million MWh (TWh)")
ax.grid(lw=0.3)
ax.legend(fontsize=8, ncol=2)
ax.xaxis.set_major_locator(mticker.MaxNLocator(integer=True))

plt.suptitle("Revenue & Sales\nFERC Form 1, 2015-2024", fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

### 5a. Average Electricity Price (Revenue / Sales)

In [ ]:
# Compute average price as revenue/sales for residential and total retail
# Filter out utilities with unreliable sales_mwh (T&D-only in deregulated states)
fig, axes = plt.subplots(1, 2, figsize=(20, 8))

for ax, cls in zip(axes, ["Residential", "Total Retail"]):
    cls_data = rev_class[rev_class["customer_class"] == cls]
    for uid in UTIL_IDS:
        df_u = cls_data[cls_data["utility_id_ferc1"] == uid].sort_values("report_year")
        df_u = df_u[df_u["sales_mwh"] >= MIN_RETAIL_MWH]
        if len(df_u) > 0:
            avg_price = df_u["dollar_value"] / df_u["sales_mwh"]  # $/MWh
            ax.plot(df_u["report_year"], avg_price * 0.1,  # convert to cents/kWh
                    color=UTIL_COLORS[uid], lw=2.5, label=UTIL_NAMES[uid])
    ax.set_title(f"Average {cls} Electricity Price", fontweight="bold")
    ax.set_xlabel("Year")
    ax.set_ylabel("2024 cents/kWh")
    ax.grid(lw=0.3)
    ax.legend(fontsize=8, ncol=2)
    ax.xaxis.set_major_locator(mticker.MaxNLocator(integer=True))

plt.suptitle("Average Electricity Prices (Real 2024 $)\nFERC Form 1 Schedule 300, 2015-2024", fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

## 6. Income Statement Analysis

Key metrics from Schedule 114: operating revenues, total expenses, net operating income,
depreciation, and taxes.

In [ ]:
# Key income statement items
INCOME_ITEMS = {
    "operating_revenues": "Operating Revenues",
    "utility_operating_expenses": "Operating Expenses",
    "net_utility_operating_income": "Net Operating Income",
    "depreciation_expense": "Depreciation",
    "income_taxes_operating_income": "Operating Income Taxes",
    "maintenance_expense": "Maintenance",
    "operation_expense": "Operations",
    "net_income_loss": "Net Income",
}

inc_items = inc[
    (inc["income_type"].isin(INCOME_ITEMS.keys())) &
    (inc["utility_type"] == "electric")
].copy()
inc_items["item"] = inc_items["income_type"].map(INCOME_ITEMS)
inc_items["utility_name"] = inc_items["utility_id_ferc1"].map(UTIL_NAMES)
inc_items["state"] = inc_items["utility_id_ferc1"].map(UTIL_STATES)

inc_pivot = inc_items.pivot_table(
    index=["utility_id_ferc1", "utility_name", "state", "report_year"],
    columns="item",
    values="dollar_value",
    aggfunc="sum"
).reset_index()

print(f"Income statement data: {len(inc_pivot)} rows")
inc_pivot.head()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(24, 8))

for ax, item in zip(axes, ["Operating Revenues", "Operating Expenses", "Net Operating Income"]):
    for uid in UTIL_IDS:
        df_u = inc_pivot[inc_pivot["utility_id_ferc1"] == uid].sort_values("report_year")
        if item in df_u.columns and df_u[item].notna().any():
            ax.plot(df_u["report_year"], df_u[item] / 1e9,
                    color=UTIL_COLORS[uid], lw=2.5, label=UTIL_NAMES[uid])
    ax.set_title(item, fontweight="bold")
    ax.set_xlabel("Year")
    ax.set_ylabel("Billion 2024 $")
    ax.grid(lw=0.3)
    ax.xaxis.set_major_locator(mticker.MaxNLocator(integer=True))

axes[0].legend(bbox_to_anchor=(0, -0.25), loc="upper left", ncol=3, fontsize=9)
plt.suptitle("Income Statement Highlights (Real 2024 $)\nFERC Form 1 Schedule 114, 2015-2024", fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

### 6a. Implied Return on Rate Base

Comparing two methods:
- **Left**: NOI / Total Electric Plant in Service (gross plant proxy — overstates denominator)
- **Right**: NOI / FERC Rate Base (actual — accounts for depreciation, ADIT, working capital)

In [ ]:
# Merge net operating income with total plant in service
noi = inc_pivot[["utility_id_ferc1", "report_year", "Net Operating Income"]].dropna()

# Method 1: Plant-in-service proxy (original)
total_plant = plant_pivot[
    plant_pivot["Total Electric Plant"].notna()
][["utility_id_ferc1", "report_year", "Total Electric Plant"]]

ror_df = noi.merge(total_plant, on=["utility_id_ferc1", "report_year"])
ror_df["implied_ror"] = ror_df["Net Operating Income"] / ror_df["Total Electric Plant"] * 100
ror_df["utility_name"] = ror_df["utility_id_ferc1"].map(UTIL_NAMES)

# Method 2: Actual rate base
total_rb_series = rb_pivot[rb_pivot["Total Rate Base"].notna()][
    ["utility_id_ferc1", "report_year", "Total Rate Base"]
]
ror_rb_df = noi.merge(total_rb_series, on=["utility_id_ferc1", "report_year"])
ror_rb_df = ror_rb_df[ror_rb_df["Total Rate Base"] > 0]
ror_rb_df["implied_ror_rb"] = (
    ror_rb_df["Net Operating Income"] / ror_rb_df["Total Rate Base"] * 100
)
ror_rb_df["utility_name"] = ror_rb_df["utility_id_ferc1"].map(UTIL_NAMES)

fig, axes = plt.subplots(1, 2, figsize=(24, 8), sharey=True)

# Left: using plant in service proxy
ax = axes[0]
for uid in UTIL_IDS:
    df_u = ror_df[ror_df["utility_id_ferc1"] == uid].sort_values("report_year")
    if len(df_u) > 0:
        ax.plot(df_u["report_year"], df_u["implied_ror"],
                color=UTIL_COLORS[uid], lw=2.5, marker="o", markersize=4,
                label=UTIL_NAMES[uid])
ax.set_title("Using Plant in Service (Proxy)", fontweight="bold")
ax.set_xlabel("Year")
ax.set_ylabel("%")
ax.grid(lw=0.3)
ax.legend(fontsize=8, ncol=2, loc="upper right")
ax.xaxis.set_major_locator(mticker.MaxNLocator(integer=True))

# Right: using actual rate base
ax = axes[1]
for uid in UTIL_IDS:
    df_u = ror_rb_df[ror_rb_df["utility_id_ferc1"] == uid].sort_values("report_year")
    if len(df_u) > 0:
        ax.plot(df_u["report_year"], df_u["implied_ror_rb"],
                color=UTIL_COLORS[uid], lw=2.5, marker="o", markersize=4,
                label=UTIL_NAMES[uid])
ax.set_title("Using FERC Rate Base (Actual)", fontweight="bold")
ax.set_xlabel("Year")
ax.grid(lw=0.3)
ax.legend(fontsize=8, ncol=2, loc="upper right")
ax.xaxis.set_major_locator(mticker.MaxNLocator(integer=True))

plt.suptitle(
    "Implied Return on Rate Base: NOI / Rate Base (%)\nFERC Form 1, 2015-2024",
    fontweight="bold", y=1.02,
)
plt.tight_layout()
plt.show()

## 7. Depreciation by Plant Function

Schedule 336 breaks depreciation down by plant function (production, transmission, distribution).

In [ ]:
# Group depreciation plant_function into broad categories
DEP_MAP = {
    "steam_production": "Production",
    "nuclear_production": "Production",
    "hydraulic_production_conventional": "Production",
    "hydraulic_production_pumped_storage": "Production",
    "other_production": "Production",
    "transmission": "Transmission",
    "distribution": "Distribution",
    "general": "General",
    "total": "Total",
}

dep_cat = dep[dep["plant_function"].isin(DEP_MAP.keys())].copy()
dep_cat["category"] = dep_cat["plant_function"].map(DEP_MAP)
dep_cat["utility_name"] = dep_cat["utility_id_ferc1"].map(UTIL_NAMES)
dep_cat["state"] = dep_cat["utility_id_ferc1"].map(UTIL_STATES)

# Total depreciation only (ferc_account_label for total depreciation charges)
dep_total = dep_cat.groupby(
    ["utility_id_ferc1", "utility_name", "state", "report_year", "category"]
)["dollar_value"].sum().reset_index()

dep_pivot = dep_total.pivot_table(
    index=["utility_id_ferc1", "utility_name", "state", "report_year"],
    columns="category",
    values="dollar_value",
    aggfunc="sum"
).reset_index()

print(f"Depreciation data: {len(dep_pivot)} rows")
dep_pivot.head()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(24, 8))

for ax, cat in zip(axes, ["Production", "Transmission", "Distribution"]):
    for uid in UTIL_IDS:
        df_u = dep_pivot[dep_pivot["utility_id_ferc1"] == uid].sort_values("report_year")
        if cat in df_u.columns and df_u[cat].notna().any():
            ax.plot(df_u["report_year"], df_u[cat] / 1e6,
                    color=UTIL_COLORS[uid], lw=2.5, label=UTIL_NAMES[uid])
    ax.set_title(f"{cat} Depreciation", fontweight="bold")
    ax.set_xlabel("Year")
    ax.set_ylabel("Million $")
    ax.grid(lw=0.3)
    ax.xaxis.set_major_locator(mticker.MaxNLocator(integer=True))

axes[0].legend(bbox_to_anchor=(0, -0.25), loc="upper left", ncol=3, fontsize=9)
plt.suptitle("Depreciation by Plant Function\nFERC Form 1 Schedule 336, 2015-2024", fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

## 8. Year-over-Year Growth Rates

In [ ]:
def compute_yoy_growth(df, uid_col, year_col, value_col):
    """Compute year-over-year percentage growth for each utility."""
    results = []
    for uid in df[uid_col].unique():
        df_u = df[df[uid_col] == uid].sort_values(year_col)
        vals = df_u[value_col].values
        years = df_u[year_col].values
        if len(vals) > 1:
            growth = (vals[1:] - vals[:-1]) * 100 / np.abs(vals[:-1])
            for y, g in zip(years[1:], growth):
                results.append({uid_col: uid, year_col: y, "growth_pct": g})
    return pd.DataFrame(results)

# Growth in total revenue
rev_growth = compute_yoy_growth(total_rev, "utility_id_ferc1", "report_year", "dollar_value")

fig, axes = plt.subplots(1, 2, figsize=(20, 8))

# Revenue growth
ax = axes[0]
for uid in UTIL_IDS:
    df_u = rev_growth[rev_growth["utility_id_ferc1"] == uid].sort_values("report_year")
    if len(df_u) > 0:
        ax.plot(df_u["report_year"], df_u["growth_pct"],
                color=UTIL_COLORS[uid], lw=2, marker="o", markersize=3,
                label=UTIL_NAMES[uid])
ax.axhline(y=0, color="black", lw=0.8)
ax.set_title("Revenue YoY Growth", fontweight="bold")
ax.set_xlabel("Year")
ax.set_ylabel("%")
ax.grid(lw=0.3)
ax.legend(fontsize=7, ncol=2)
ax.xaxis.set_major_locator(mticker.MaxNLocator(integer=True))

# Distribution O&M growth
dist_opex = opex_pivot[["utility_id_ferc1", "report_year", "Distribution"]].dropna()
dist_growth = compute_yoy_growth(dist_opex, "utility_id_ferc1", "report_year", "Distribution")
ax = axes[1]
for uid in UTIL_IDS:
    df_u = dist_growth[dist_growth["utility_id_ferc1"] == uid].sort_values("report_year")
    if len(df_u) > 0:
        ax.plot(df_u["report_year"], df_u["growth_pct"],
                color=UTIL_COLORS[uid], lw=2, marker="o", markersize=3,
                label=UTIL_NAMES[uid])
ax.axhline(y=0, color="black", lw=0.8)
ax.set_title("Distribution O&M YoY Growth", fontweight="bold")
ax.set_xlabel("Year")
ax.set_ylabel("%")
ax.grid(lw=0.3)
ax.legend(fontsize=7, ncol=2)
ax.xaxis.set_major_locator(mticker.MaxNLocator(integer=True))

plt.suptitle("Year-over-Year Growth Rates\nFERC Form 1, 2015-2024", fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

## 9. Revenue Requirement Decomposition

Approximate revenue requirement = O&M + Depreciation + Taxes + Return on Rate Base.
We combine income statement data to decompose what drives electricity costs.

In [ ]:
# Build revenue requirement components from income statement
rr_components = inc_pivot.copy()
rr_components = rr_components[rr_components["report_year"].isin(YEARS)]

# Revenue requirement stacked area per utility (largest utilities)
# Pick the biggest utility per state for clarity
largest_by_state = {}
for st, uids in STATE_UTILS.items():
    rev_by_uid = total_rev.groupby("utility_id_ferc1")["dollar_value"].sum()
    best_uid = max(uids, key=lambda u: rev_by_uid.get(u, 0))
    largest_by_state[st] = best_uid

fig, axes = plt.subplots(len(largest_by_state), 1, figsize=(14, 5 * len(largest_by_state)), sharex=True)

for ax, (st, uid) in zip(axes, largest_by_state.items()):
    df_u = rr_components[rr_components["utility_id_ferc1"] == uid].sort_values("report_year")
    if len(df_u) == 0:
        continue

    # Available components
    components = []
    labels = []
    colors = []
    comp_map = [
        ("Operations", "tab:blue"),
        ("Maintenance", "tab:cyan"),
        ("Depreciation", "tab:orange"),
        ("Operating Income Taxes", "tab:red"),
        ("Net Operating Income", "tab:green"),
    ]
    for comp_name, comp_color in comp_map:
        if comp_name in df_u.columns and df_u[comp_name].notna().any():
            components.append(df_u[comp_name].fillna(0).values / 1e9)
            labels.append(comp_name)
            colors.append(comp_color)

    if components:
        ax.stackplot(df_u["report_year"].values, *components,
                     labels=labels, colors=colors, alpha=0.8)
        # Overlay total revenue
        if "Operating Revenues" in df_u.columns:
            ax.plot(df_u["report_year"], df_u["Operating Revenues"] / 1e9,
                    color="black", lw=3, linestyle="--", label="Total Revenue")

    ax.set_title(f"{UTIL_NAMES[uid]} ({STATE_NAMES[st]})", fontweight="bold")
    ax.set_ylabel("Billion 2024 $")
    ax.legend(loc="upper left", fontsize=9)
    ax.grid(lw=0.3)

axes[-1].set_xlabel("Year")
plt.suptitle("Revenue Requirement Decomposition (Real 2024 $)\nFERC Form 1 Schedule 114", fontweight="bold", y=1.01)
plt.tight_layout()
plt.show()

## 10. Cost per MWh Analysis

Divide O&M costs by total retail sales (MWh) to get cost intensity.

In [ ]:
# Get retail sales MWh from schedule 300 (filter out unreliable low-MWh utilities)
retail_sales = rev_class[
    (rev_class["customer_class"] == "Total Retail")
    & (rev_class["sales_mwh"] >= MIN_RETAIL_MWH)
][["utility_id_ferc1", "report_year", "sales_mwh"]].rename(
    columns={"sales_mwh": "retail_mwh"}
)

# Merge with O&M by category
cost_per_mwh = opex_pivot.merge(retail_sales, on=["utility_id_ferc1", "report_year"])

fig, axes = plt.subplots(2, 2, figsize=(24, 16), sharey=False)

plot_cats = [
    ("Power Supply", axes[0, 0]),
    ("Purchased Power", axes[0, 1]),
    ("Transmission", axes[1, 0]),
    ("Distribution", axes[1, 1]),
]

for cat, ax in plot_cats:
    for uid in UTIL_IDS:
        df_u = cost_per_mwh[cost_per_mwh["utility_id_ferc1"] == uid].sort_values("report_year")
        if cat in df_u.columns and df_u[cat].notna().any() and (df_u["retail_mwh"] > 0).any():
            cost_intensity = df_u[cat] / df_u["retail_mwh"]  # $/MWh
            ax.plot(df_u["report_year"], cost_intensity,
                    color=UTIL_COLORS[uid], lw=2.5, label=UTIL_NAMES[uid])
    subtitle = "(Gen O&M + Purchased Power)" if cat == "Power Supply" else ""
    ax.set_title(f"{cat} per MWh{chr(10)}{subtitle}".strip(), fontweight="bold")
    ax.set_xlabel("Year")
    ax.set_ylabel("2024 $/MWh")
    ax.grid(lw=0.3)
    ax.xaxis.set_major_locator(mticker.MaxNLocator(integer=True))

axes[0, 0].legend(bbox_to_anchor=(0, -0.25), loc="upper left", ncol=3, fontsize=9)
plt.suptitle("O&M Cost Intensity (Real 2024 $/MWh Retail Sales)\nFERC Form 1, 2015-2024", fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

## 11. Summary Table: Key Metrics by Utility

In [ ]:
latest_year = max(YEARS) if max(YEARS) in opex_pivot["report_year"].values else max(YEARS) - 1

summary_rows = []
for uid in UTIL_IDS:
    row = {"Utility": UTIL_NAMES[uid], "State": UTIL_STATES[uid]}

    # Latest year revenue
    tr = total_rev[(total_rev["utility_id_ferc1"] == uid) & (total_rev["report_year"] == latest_year)]
    row["Revenue ($B)"] = tr["dollar_value"].sum() / 1e9 if len(tr) > 0 else np.nan

    # Latest year O&M
    op = opex_pivot[(opex_pivot["utility_id_ferc1"] == uid) & (opex_pivot["report_year"] == latest_year)]
    for cat in ["Generation", "Purchased Power", "Power Supply", "Transmission", "Distribution"]:
        row[f"{cat} O&M ($M)"] = op[cat].sum() / 1e6 if (cat in op.columns and len(op) > 0) else np.nan

    # Total plant
    pl = plant_pivot[(plant_pivot["utility_id_ferc1"] == uid) & (plant_pivot["report_year"] == latest_year)]
    row["Total Plant ($B)"] = pl["Total Electric Plant"].sum() / 1e9 if ("Total Electric Plant" in pl.columns and len(pl) > 0) else np.nan

    # Retail MWh
    rs = retail_sales[(retail_sales["utility_id_ferc1"] == uid) & (retail_sales["report_year"] == latest_year)]
    row["Retail Sales (TWh)"] = rs["retail_mwh"].sum() / 1e6 if len(rs) > 0 else np.nan

    # Implied ROR
    rr = ror_df[(ror_df["utility_id_ferc1"] == uid) & (ror_df["report_year"] == latest_year)]
    row["Implied ROR (%)"] = rr["implied_ror"].values[0] if len(rr) > 0 else np.nan

    summary_rows.append(row)

summary = pd.DataFrame(summary_rows)
summary = summary.round(2)
print(f"\nSummary for {latest_year}:")
summary

## 12. Cumulative % Change: 2015 to Latest Year

In [ ]:
pct_rows = []
for uid in UTIL_IDS:
    row = {"Utility": UTIL_NAMES[uid], "State": UTIL_STATES[uid]}

    # O&M categories (now including Purchased Power and Power Supply)
    for cat in ["Generation", "Purchased Power", "Power Supply", "Transmission", "Distribution"]:
        base = opex_pivot[(opex_pivot["utility_id_ferc1"] == uid) & (opex_pivot["report_year"] == BASE_YEAR)]
        latest = opex_pivot[(opex_pivot["utility_id_ferc1"] == uid) & (opex_pivot["report_year"] == latest_year)]
        if len(base) > 0 and len(latest) > 0 and cat in base.columns:
            b = base[cat].sum()
            l = latest[cat].sum()
            row[f"{cat} O&M %chg"] = ((l - b) / abs(b) * 100) if b != 0 else np.nan

    # Plant in service
    for cat in ["Production", "Transmission", "Distribution"]:
        base = plant_pivot[(plant_pivot["utility_id_ferc1"] == uid) & (plant_pivot["report_year"] == BASE_YEAR)]
        latest = plant_pivot[(plant_pivot["utility_id_ferc1"] == uid) & (plant_pivot["report_year"] == latest_year)]
        if len(base) > 0 and len(latest) > 0 and cat in base.columns:
            b = base[cat].sum()
            l = latest[cat].sum()
            row[f"{cat} Plant %chg"] = ((l - b) / abs(b) * 100) if b != 0 else np.nan

    # Revenue
    base_r = total_rev[(total_rev["utility_id_ferc1"] == uid) & (total_rev["report_year"] == BASE_YEAR)]
    latest_r = total_rev[(total_rev["utility_id_ferc1"] == uid) & (total_rev["report_year"] == latest_year)]
    if len(base_r) > 0 and len(latest_r) > 0:
        b = base_r["dollar_value"].sum()
        l = latest_r["dollar_value"].sum()
        row["Revenue %chg"] = ((l - b) / abs(b) * 100) if b != 0 else np.nan

    pct_rows.append(row)

pct_df = pd.DataFrame(pct_rows).round(1)
print(f"\nCumulative % Change: {BASE_YEAR} to {latest_year}")
pct_df

## 13. Cross-State Comparison: Average Price by State

In [ ]:
# Aggregate revenue and sales by state (exclude utilities with unreliable sales_mwh)
rev_class_state = rev_class[rev_class["sales_mwh"] >= MIN_RETAIL_MWH].copy()
state_retail = rev_class_state[
    rev_class_state["customer_class"] == "Total Retail"
].groupby(["state", "report_year"]).agg(
    total_revenue=("dollar_value", "sum"),
    total_mwh=("sales_mwh", "sum")
).reset_index()

state_retail["avg_price_cents_kwh"] = (state_retail["total_revenue"] / state_retail["total_mwh"]) * 0.1

fig, ax = plt.subplots(figsize=(12, 8))
for st in STATE_COLORS:
    df_s = state_retail[state_retail["state"] == st].sort_values("report_year")
    if len(df_s) > 0 and df_s["avg_price_cents_kwh"].notna().any():
        ax.plot(df_s["report_year"], df_s["avg_price_cents_kwh"],
                color=STATE_COLORS[st], lw=4, label=STATE_NAMES[st])

ax.set_title("Average Retail Electricity Price by State (IOU Aggregate)\nFERC Form 1, 2015-2024", fontweight="bold")
ax.set_xlabel("Year")
ax.set_ylabel("2024 cents/kWh")
ax.grid(lw=0.3)
ax.legend(fontsize=14)
ax.xaxis.set_major_locator(mticker.MaxNLocator(integer=True))
plt.tight_layout()
plt.show()

## 14. T&D Share of Total Plant

In [ ]:
# T&D as share of total plant
td_share = plant_pivot.copy()
if "Transmission" in td_share.columns and "Distribution" in td_share.columns and "Total Electric Plant" in td_share.columns:
    td_share["T&D_share"] = (td_share["Transmission"].fillna(0) + td_share["Distribution"].fillna(0)) / td_share["Total Electric Plant"] * 100

    fig, ax = plt.subplots(figsize=(14, 8))
    for uid in UTIL_IDS:
        df_u = td_share[td_share["utility_id_ferc1"] == uid].sort_values("report_year")
        if len(df_u) > 0 and df_u["T&D_share"].notna().any():
            ax.plot(df_u["report_year"], df_u["T&D_share"],
                    color=UTIL_COLORS[uid], lw=2.5, label=UTIL_NAMES[uid])

    ax.set_title("T&D Share of Total Electric Plant in Service\nFERC Form 1 Schedule 204", fontweight="bold")
    ax.set_xlabel("Year")
    ax.set_ylabel("% of Total")
    ax.grid(lw=0.3)
    ax.legend(fontsize=9, ncol=3)
    ax.xaxis.set_major_locator(mticker.MaxNLocator(integer=True))
    plt.tight_layout()
    plt.show()

## 15. Revenue per MWh by State (Revenue Requirement Proxy)

In [ ]:
# Total revenue / total retail MWh by state
# Only include revenue from utilities with reliable retail sales data
utils_with_sales = retail_sales["utility_id_ferc1"].unique()
total_rev_state = (
    total_rev[total_rev["utility_id_ferc1"].isin(utils_with_sales)]
    .groupby(["state", "report_year"])["dollar_value"]
    .sum()
    .reset_index()
)
# retail_sales already filtered to MIN_RETAIL_MWH above
retail_mwh_state = retail_sales.copy()
retail_mwh_state["state"] = retail_mwh_state["utility_id_ferc1"].map(UTIL_STATES)
retail_mwh_state = retail_mwh_state.groupby(["state", "report_year"])["retail_mwh"].sum().reset_index()

rev_mwh_state = total_rev_state.merge(retail_mwh_state, on=["state", "report_year"])
rev_mwh_state["rev_per_mwh"] = rev_mwh_state["dollar_value"] / rev_mwh_state["retail_mwh"]

# Normalize
fig, axes = plt.subplots(1, 2, figsize=(20, 8))

ax = axes[0]
for st in STATE_COLORS:
    df_s = rev_mwh_state[rev_mwh_state["state"] == st].sort_values("report_year")
    if len(df_s) > 0:
        ax.plot(df_s["report_year"], df_s["rev_per_mwh"],
                color=STATE_COLORS[st], lw=4, label=STATE_NAMES[st])
ax.set_title("Revenue per MWh (Real 2024 $)", fontweight="bold")
ax.set_xlabel("Year")
ax.set_ylabel("2024 $/MWh")
ax.grid(lw=0.3)
ax.legend(fontsize=14)
ax.xaxis.set_major_locator(mticker.MaxNLocator(integer=True))

ax = axes[1]
for st in STATE_COLORS:
    df_s = rev_mwh_state[rev_mwh_state["state"] == st].sort_values("report_year")
    if len(df_s) > 0:
        base_val = df_s[df_s["report_year"] == BASE_YEAR]["rev_per_mwh"]
        if not base_val.empty and base_val.item() > 0:
            ax.plot(df_s["report_year"], df_s["rev_per_mwh"] / base_val.item(),
                    color=STATE_COLORS[st], lw=4, label=STATE_NAMES[st])
ax.axhline(y=1, linestyle="dashed", color="grey", lw=1)
ax.set_title(f"Revenue per MWh Normalized ({BASE_YEAR} = 1.0)", fontweight="bold")
ax.set_xlabel("Year")
ax.set_ylabel(f"Relative to {BASE_YEAR}")
ax.grid(lw=0.3)
ax.legend(fontsize=14)
ax.xaxis.set_major_locator(mticker.MaxNLocator(integer=True))

plt.suptitle("Revenue Requirement per MWh by State (Real 2024 $)\nFERC Form 1, 2015-2024", fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()